In [46]:
import os
import json
import csv

In [83]:
from cmat.trait_mapping.ols import is_current_and_in_ontology

### Investigations for cleaning up trait mappings

In [87]:
def print_set(s, n=10):
    # Helper function to peek at large sets
    i = 0
    for x in s:
        print(x)
        i += 1
        if i > n:
            break

* How many mappings in the latest_mappings file are used in latest evidence strings / latest ClinVar?

In [22]:
# Get all trait names with EFO mappings from 2025.12 evidence
# Note this isn't the same as all trait names in ClinVar with EFO mappings - these records might be filtered out of the evidence for other reasons

# To easily handle multiples we define a "mapping" as a pair: (trait_name, ontology_id)
# This mirrors how they're counted in the tsv files where each row is a mapping.
mappings_in_evidence = set()
with open(os.path.join(os.getenv('BATCH_ROOT'), 'batch-2025-12/evidence_strings/evidence_strings.json')) as f:
    for line in f:
        evidence = json.loads(line.strip())
        if evidence.get('diseaseFromSourceMappedId') and evidence.get('diseaseFromSource'):
            mappings_in_evidence.add((evidence.get('diseaseFromSource').lower(), evidence.get('diseaseFromSourceMappedId')))

In [23]:
len(mappings_in_evidence)

12638

In [15]:
!wc ${BATCH_ROOT}/manual_curation/latest_mappings.tsv

  15975  142159 1860681 /nfs/production/keane/eva/opentargets/manual_curation/latest_mappings.tsv


In [21]:
# Percentage of mappings being used in latest evidence
12638/15975

0.7911111111111111

* How many mappings are being used but are bypassing curation completely (high confidence from Zooma or exact match from OLS)?
* Are there any mappings that are used in evidence strings but aren’t in the most recent curated and automated mappings?

In [78]:
# Check where these mappings come from: most recent automated mappings, most recent manual curation, or neither

# Use a dict so we can hash on (trait_name, ontology_id), but keep the URI to check obsoleteness
automated_mappings = {}
with open(os.path.join(os.getenv('BATCH_ROOT'), 'manual_curation/2025-10-02/automated_trait_mappings.tsv')) as f:
    for line in f:
        trait_name, ontology_uri, ontology_label = line.strip().split('\t')
        automated_mappings[(trait_name.lower(), ontology_uri.split('/')[-1])] = ontology_uri

In [79]:
len(automated_mappings)

11075

In [80]:
curated_mappings = {}
with open(os.path.join(os.getenv('BATCH_ROOT'), 'manual_curation/2025-10-02/finished_curation_spreadsheet.csv')) as f:
    reader = csv.reader(f, dialect='excel')
    # skip header
    next(reader)
    next(reader)
    for row in reader:
        if row[5] == 'DONE' and row[0] and row[7]:
            curated_mappings[(row[7].lower(), row[0].split('/')[-1])] = row[0]

In [81]:
len(curated_mappings)

371

In [82]:
# Mappings carried over from previous (not in automated or curated)
15975 - (11075 + 371)

4529

In [84]:
mappings_used_in_automated = set()
mappings_used_in_curated = set()
mappings_used_in_neither = set()  # these are "previous mappings" only
mappings_used_in_both = set()  # sanity check - this should never happen!

for mapping in mappings_in_evidence:
    if mapping in automated_mappings and mapping not in curated_mappings:
        mappings_used_in_automated.add(mapping)
    if mapping not in automated_mappings and mapping in curated_mappings:
        mappings_used_in_curated.add(mapping)
    if mapping in automated_mappings and mapping in curated_mappings:
        mappings_used_in_both.add(mapping)
    if mapping not in automated_mappings and mapping not in curated_mappings:
        mappings_used_in_neither.add(mapping)

In [85]:
len(mappings_used_in_automated)

10594

In [86]:
len(mappings_used_in_curated)

355

In [72]:
len(mappings_in_neither)

1689

In [66]:
len(mappings_in_both)

0

In [73]:
# Confirm these add up to the number of mappings used in the evidence
10594 + 355 + 1689

12638

In [89]:
# How many in each of these categories are obsolete?
n_obsolete_automated = 0
for mapping in mappings_used_in_automated:
    if not is_current_and_in_ontology(automated_mappings[mapping]):
        n_obsolete_automated += 1        

In [90]:
n_obsolete_curated = 0
for mapping in mappings_used_in_curated:
    if not is_current_and_in_ontology(curated_mappings[mapping]):
        n_obsolete_curated += 1

In [95]:
n_obsolete_automated

13

In [96]:
n_obsolete_curated

2

In [93]:
# For previous mappings, need to refer to latest_mappings file instead
all_latest_mappings = {}
with open(os.path.join(os.getenv('BATCH_ROOT'), 'manual_curation/2025-10-02/trait_names_to_ontology_mappings.tsv')) as f:
    # skip 3 lines
    next(f)
    next(f)
    next(f)
    for line in f:
        trait_name, ontology_uri, ontology_label = line.strip().split('\t')
        all_latest_mappings[(trait_name.lower(), ontology_uri.split('/')[-1])] = ontology_uri

In [92]:
n_obsolete_previous = 0
for mapping in mappings_in_neither:
    if not is_current_and_in_ontology(all_latest_mappings[mapping]):
        n_obsolete_previous += 1

KeyError: ('focal facial dermal dysplasia type iii', 'Orphanet_1807')

```
KeyError: ('focal facial dermal dysplasia type iii', 'Orphanet_1807')
```
This mapping is indeed not in latest mappings, due to [this quirk](https://github.com/EBIvariation/CMAT/issues/384) about how we annotate traits in ClinVar.

```
$ grep -i 'focal facial dermal dysplasia type iii' trait_names_to_ontology_mappings.tsv
focal facial dermal dysplasia type iii  http://purl.obolibrary.org/obo/MONDO_0009203    focal facial dermal dysplasia type III
congenital ectodermal dysplasia of face http://www.orpha.net/ORDO/Orphanet_1807 Focal facial dermal dysplasia type III
focal facial dermal dysplasia 3, setleis type   http://www.orpha.net/ORDO/Orphanet_1807 Focal facial dermal dysplasia type III

$ grep -i 'focal facial dermal dysplasia type iii' batch-2025-12/evidence_strings/evidence_strings.json
<json abbreviated>
{..."cohortPhenotypes": ["BITEMPORAL FORCEPS MARKS SYNDROME", "FFDD type 2", "FOCAL FACIAL DERMAL DYSPLASIA, TYPE II", "Focal facial dermal dysplasia 3", "Focal facial dermal dysplasia 3, Setleis type", "Focal facial dermal dysplasia type III", "SETLEIS SYNDROME"],
    "diseaseFromSource": "Focal facial dermal dysplasia type III",
    "diseaseFromSourceId": "C1744559", 
    "diseaseFromSourceMappedId": "Orphanet_1807", ...}
{..."cohortPhenotypes": ["BITEMPORAL FORCEPS MARKS SYNDROME", "FFDD type 2", "FOCAL FACIAL DERMAL DYSPLASIA, TYPE II", "Focal facial dermal dysplasia 3", "Focal facial dermal dysplasia 3, Setleis type", "Focal facial dermal dysplasia type III", "SETLEIS SYNDROME"],
    "diseaseFromSource": "Focal facial dermal dysplasia type III", 
    "diseaseFromSourceId": "C1744559",
    "diseaseFromSourceMappedId": "Orphanet_398166", ...}
{..."cohortPhenotypes": ["BITEMPORAL FORCEPS MARKS SYNDROME", "FFDD type 2", "FOCAL FACIAL DERMAL DYSPLASIA, TYPE II", "Focal facial dermal dysplasia 3", "Focal facial dermal dysplasia 3, Setleis type", "Focal facial dermal dysplasia type III", "SETLEIS SYNDROME"],
    "diseaseFromSource": "Focal facial dermal dysplasia type III",
    "diseaseFromSourceId": "C1744559",
    "diseaseFromSourceMappedId": "MONDO_0009203", ...}
<more results>

# The other two mappings appear above, check the third
$ grep -i 'Orphanet_398166' trait_names_to_ontology_mappings.tsv
congenital ectodermal dysplasia of face http://www.orpha.net/ORDO/Orphanet_398166       Focal facial dermal dysplasia
focal facial dermal dysplasia 3, setleis type   http://www.orpha.net/ORDO/Orphanet_398166       Focal facial dermal dysplasia
```
Note that both [Orphanet_1807](https://www.ebi.ac.uk/ols4/ontologies/efo/classes/http%253A%252F%252Fwww.orpha.net%252FORDO%252FOrphanet_1807) and [Orphanet_398166](https://www.ebi.ac.uk/ols4/ontologies/efo/classes/http%253A%252F%252Fwww.orpha.net%252FORDO%252FOrphanet_398166) are deprecated in EFO, which is what Issue 384 is about.

This issue is a separate one from what we're investigating (though not using previous mappings would mitigate this as well). For the purposes of this investigation, the point is that these `(diseaseFromSource, diseaseFromSourceMappedId)` tuples are not actually the mappings we should be using for counts!

* Are there mapping that used to be high confidence in automated mapping and subsequently was mapped with low confidence